
# Advanced Bahdanau Attention Mechanism — PyTorch

This notebook implements an **advanced sequence-to-sequence model with Bahdanau (additive) attention** for a synthetic sequence-reversal task.

### Included
- Encoder LSTM
- Bahdanau additive attention from scratch
- Decoder LSTM
- Teacher forcing
- Masked attention for variable-length sequences
- Training/validation loop
- Attention-weight visualization
- Greedy decoding
- BLEU-style token accuracy evaluation
- Gradient clipping and learning-rate scheduling

> **Note:** The correct spelling is usually **Bahdanau Attention**, named after Dzmitry Bahdanau et al.


In [ ]:

# 1. Imports and reproducibility
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:

# 2. Vocabulary and synthetic dataset
PAD, SOS, EOS = 0, 1, 2
FIRST_TOKEN = 3
VOCAB_SIZE = 30

class ReverseSequenceDataset(Dataset):
    def __init__(self, n_samples=8000, min_len=4, max_len=12):
        self.samples = []
        for _ in range(n_samples):
            length = random.randint(min_len, max_len)
            seq = [random.randint(FIRST_TOKEN, VOCAB_SIZE - 1) for _ in range(length)]
            target = list(reversed(seq))
            self.samples.append((seq, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    src, tgt = zip(*batch)
    src_lengths = torch.tensor([len(x) for x in src], dtype=torch.long)

    max_src = max(map(len, src))
    max_tgt = max(map(len, tgt)) + 1  # + EOS

    src_batch = torch.full((len(batch), max_src), PAD, dtype=torch.long)
    tgt_batch = torch.full((len(batch), max_tgt + 1), PAD, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src, tgt)):
        src_batch[i, :len(s)] = torch.tensor(s)
        target = t + [EOS]
        tgt_batch[i, :len(target)] = torch.tensor(target)

    # Decoder input: SOS + target[:-1]
    decoder_input = torch.full_like(tgt_batch, PAD)
    decoder_input[:, 0] = SOS
    decoder_input[:, 1:] = tgt_batch[:, :-1]

    return src_batch, src_lengths, decoder_input, tgt_batch

train_ds = ReverseSequenceDataset(8000)
val_ds = ReverseSequenceDataset(1000)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, collate_fn=collate_fn)

print("Training examples:", len(train_ds))
print("Validation examples:", len(val_ds))



## 3. Bahdanau Additive Attention

For decoder state \(s_t\) and encoder output \(h_i\):

\[
e_{t,i}=v_a^T	anh(W_s s_t + W_h h_i)
\]

The attention weights are:

\[
lpha_{t,i}=	ext{softmax}(e_{t,i})
\]

and the context vector is:

\[
c_t=\sum_i lpha_{t,i}h_i
\]

The implementation below explicitly masks padded encoder positions.


In [ ]:

# 4. Bahdanau Attention module
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_proj = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.decoder_proj = nn.Linear(decoder_dim, attention_dim, bias=False)
        self.energy = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, mask):
        # decoder_hidden: [B, decoder_dim]
        # encoder_outputs: [B, S, encoder_dim]
        enc = self.encoder_proj(encoder_outputs)                    # [B,S,A]
        dec = self.decoder_proj(decoder_hidden).unsqueeze(1)        # [B,1,A]

        scores = self.energy(torch.tanh(enc + dec)).squeeze(-1)     # [B,S]
        scores = scores.masked_fill(~mask, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)            # [B,S]
        context = torch.bmm(attention_weights.unsqueeze(1),
                            encoder_outputs).squeeze(1)              # [B,E]

        return context, attention_weights


In [ ]:

# 5. Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, src, lengths):
        embedded = self.dropout(self.embedding(src))

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs, batch_first=True
        )

        return outputs, hidden, cell


In [ ]:

# 6. Attention-based decoder
class AttentionDecoder(nn.Module):
    def __init__(
        self, vocab_size, embedding_dim, encoder_dim, decoder_dim,
        attention_dim, num_layers=2, dropout=0.2
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)

        self.attention = BahdanauAttention(
            encoder_dim, decoder_dim, attention_dim
        )

        self.rnn = nn.LSTM(
            embedding_dim + encoder_dim,
            decoder_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.output = nn.Linear(decoder_dim + encoder_dim + embedding_dim, vocab_size)

    def forward(self, token, hidden, cell, encoder_outputs, mask):
        # token: [B]
        embedded = self.dropout(self.embedding(token)).unsqueeze(1)  # [B,1,D]

        # Last decoder-layer hidden state drives attention.
        decoder_state = hidden[-1]
        context, weights = self.attention(
            decoder_state, encoder_outputs, mask
        )

        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        output = output.squeeze(1)
        embedded = embedded.squeeze(1)

        logits = self.output(
            torch.cat([output, context, embedded], dim=-1)
        )

        return logits, hidden, cell, weights


In [ ]:

# 7. Full Seq2Seq model
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def make_mask(self, src):
        return src != PAD

    def forward(self, src, src_lengths, decoder_input, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        target_len = decoder_input.size(1)
        vocab_size = self.decoder.output.out_features

        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)
        mask = self.make_mask(src)

        outputs = torch.zeros(
            batch_size, target_len, vocab_size, device=src.device
        )
        attention_history = []

        token = decoder_input[:, 0]

        for t in range(target_len):
            logits, hidden, cell, attn = self.decoder(
                token, hidden, cell, encoder_outputs, mask
            )

            outputs[:, t] = logits
            attention_history.append(attn)

            if t + 1 < target_len:
                teacher = random.random() < teacher_forcing_ratio
                predicted = logits.argmax(dim=-1)
                token = decoder_input[:, t + 1] if teacher else predicted

        attention_history = torch.stack(attention_history, dim=1)
        return outputs, attention_history


In [ ]:

# 8. Hyperparameters and model construction
EMBED_DIM = 128
ENC_HIDDEN = 256
DEC_HIDDEN = 256
ATTN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.25

encoder = Encoder(
    VOCAB_SIZE, EMBED_DIM, ENC_HIDDEN,
    num_layers=NUM_LAYERS, dropout=DROPOUT
)

decoder = AttentionDecoder(
    VOCAB_SIZE, EMBED_DIM, ENC_HIDDEN, DEC_HIDDEN,
    ATTN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT
)

model = Seq2SeqAttention(encoder, decoder).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")


In [ ]:

# 9. Training and evaluation utilities
def run_epoch(loader, train=True, teacher_forcing_ratio=0.5):
    model.train(train)
    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0

    for src, lengths, dec_in, targets in loader:
        src, lengths = src.to(DEVICE), lengths.to(DEVICE)
        dec_in, targets = dec_in.to(DEVICE), targets.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits, _ = model(
                src, lengths, dec_in,
                teacher_forcing_ratio=teacher_forcing_ratio if train else 0.0
            )

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1)
            )

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item() * targets.numel()

        predictions = logits.argmax(dim=-1)
        valid = targets != PAD
        correct_tokens += ((predictions == targets) & valid).sum().item()
        total_tokens += valid.sum().item()

    return total_loss / len(loader.dataset), correct_tokens / total_tokens

EPOCHS = 12
best_val = float("inf")

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, True, 0.65)
    val_loss, val_acc = run_epoch(val_loader, False, 0.0)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "bahdanau_attention_best.pt")

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )


In [ ]:

# 10. Plot training curves
fig = plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Bahdanau Attention — Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

fig = plt.figure(figsize=(8, 5))
plt.plot(history["train_acc"], label="Train Token Accuracy")
plt.plot(history["val_acc"], label="Validation Token Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Bahdanau Attention — Token Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:

# 11. Greedy decoding with attention visualization
@torch.no_grad()
def predict(sequence, max_len=20):
    model.eval()

    src = torch.tensor(sequence, dtype=torch.long, device=DEVICE).unsqueeze(0)
    lengths = torch.tensor([len(sequence)], dtype=torch.long, device=DEVICE)

    encoder_outputs, hidden, cell = model.encoder(src, lengths)
    mask = src != PAD

    token = torch.tensor([SOS], dtype=torch.long, device=DEVICE)

    predictions = []
    attentions = []

    for _ in range(max_len):
        logits, hidden, cell, weights = model.decoder(
            token, hidden, cell, encoder_outputs, mask
        )
        token = logits.argmax(dim=-1)
        predicted = token.item()

        attentions.append(weights.squeeze(0).cpu().numpy())

        if predicted == EOS:
            break

        predictions.append(predicted)

    return predictions, np.array(attentions)

# Load best model
model.load_state_dict(torch.load("bahdanau_attention_best.pt", map_location=DEVICE))

example = [5, 12, 21, 8, 17, 4]
prediction, attention = predict(example)

print("Input:     ", example)
print("Expected:  ", list(reversed(example)))
print("Predicted: ", prediction)


In [ ]:

# 12. Visualize the attention matrix
def plot_attention(source, predicted, attention_matrix):
    fig = plt.figure(figsize=(10, 6))
    plt.imshow(attention_matrix, aspect="auto")
    plt.colorbar(label="Attention weight")
    plt.xticks(range(len(source)), source)
    plt.yticks(range(len(predicted)), predicted)
    plt.xlabel("Encoder input token")
    plt.ylabel("Decoder output token")
    plt.title("Bahdanau Attention Alignment")
    plt.tight_layout()
    plt.show()

if len(prediction) > 0:
    plot_attention(example, prediction, attention[:len(prediction), :len(example)])


In [ ]:

# 13. Inspect attention concentration
def attention_entropy(attention_matrix, eps=1e-12):
    p = np.clip(attention_matrix, eps, 1.0)
    return -(p * np.log(p)).sum(axis=-1)

entropy = attention_entropy(attention[:len(prediction), :len(example)])
print("Attention entropy per decoder step:")
print(np.round(entropy, 4))
print("Lower entropy generally indicates more concentrated attention.")



## 14. Key takeaways

1. **Bahdanau attention is additive attention**: encoder and decoder representations are projected into a shared attention space and combined with `tanh`.
2. **Padding is masked before softmax**, preventing padded positions from receiving probability mass.
3. **The context vector changes at every decoder step**, allowing the decoder to focus on different encoder tokens.
4. **Teacher forcing** accelerates training but is disabled during validation/inference.
5. **Gradient clipping** helps stabilize recurrent-network training.
6. The notebook is intentionally self-contained and uses a synthetic reversal task so the attention mechanism can be studied without downloading a large dataset.

### Suggested extensions
- Replace LSTM with GRU.
- Add bidirectional encoder support.
- Compare Bahdanau vs. Luong (dot-product) attention.
- Train on a real machine-translation dataset.
- Add beam search decoding.
- Add coverage attention to reduce repeated alignment.
